# MolE Pretraining: Cross-Environment Data Creation Demo

This notebook demonstrates how to create pretraining data for MolE when using **different atom environments** (e.g., different radii and/or feature settings) for the input and the prediction target.

- **Input tokens:** Atom environments with one set of parameters (e.g., radius=0, useFeatures=False)
- **Prediction targets:** Atom environments with a different set (e.g., radius=1, useFeatures=True)

This cross-environment setup is a key innovation in MolE's self-supervised pretraining.

In [1]:
import pickle
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
import torch
from collections import defaultdict

# Load vocabularies
with open('../mole/data/vocabularies/vocabulary_radius0_structural_guacamol_v1.pkl', 'rb') as f:
    vocab_input = pickle.load(f)
with open('../mole/data/vocabularies/vocabulary_radius1_functional_guacamol_v1.pkl', 'rb') as f:
    vocab_target = pickle.load(f)

# Special token IDs (corrected - vocabularies use tokens without angle brackets)
PAD_ID = vocab_input.get('PAD', 0)
MASK_ID = vocab_input.get('MASK', 1)
UNK_ID = vocab_input.get('UNK', 2)

print(f'Corrected special token IDs: PAD={PAD_ID}, MASK={MASK_ID}, UNK={UNK_ID}')

# Sample molecules
sample_molecules = {
    'Aspirin': 'CC(=O)OC1=CC=CC=C1C(=O)O',
    'Caffeine': 'CN1C=NC2=C1C(=O)N(C(=O)N2C)C',
    'Benzene': 'C1=CC=CC=C1',
    'Ethanol': 'CCO',
}


Corrected special token IDs: PAD=0, MASK=170, UNK=171


In [2]:
vocab_target

{3764344801: 1,
 3766532888: 2,
 594405804: 3,
 3205496824: 4,
 3205495869: 5,
 3764335747: 6,
 728943675: 7,
 3766532902: 8,
 614176388: 9,
 3768571846: 10,
 3764344823: 11,
 3205496507: 12,
 728919389: 13,
 3768571706: 14,
 3766532918: 15,
 3205496003: 16,
 594405818: 17,
 1541844861: 18,
 728933209: 19,
 3766532900: 20,
 614173363: 21,
 3766528779: 22,
 3205496001: 23,
 1541844807: 24,
 3205496391: 25,
 614176457: 26,
 3766528791: 27,
 3764335765: 28,
 3205496825: 29,
 606136866: 30,
 728953866: 31,
 613660003: 32,
 728933187: 33,
 614176394: 34,
 3766532953: 35,
 594910150: 36,
 614173680: 37,
 614176392: 38,
 3205496716: 39,
 3205496709: 40,
 614173298: 41,
 729164322: 42,
 728945448: 43,
 3205495832: 44,
 3766532917: 45,
 3205496734: 46,
 614176407: 47,
 614176393: 48,
 3764348914: 49,
 3208849907: 50,
 729011515: 51,
 614176451: 52,
 3766532903: 53,
 728943649: 54,
 729011661: 55,
 3205495934: 56,
 3766532898: 57,
 3764343670: 58,
 594640068: 59,
 3766523001: 60,
 3766674798: 61

In [3]:
MASK_ID

170

## 1. Define Atom Environment Extraction Functions
We will extract two sets of atom environments for each atom: one for input, one for prediction.

In [4]:
def get_atom_envs(smiles, radius=0, use_features=False):
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return None
    info = {}
    fp = AllChem.GetMorganFingerprint(
        mol,
        radius=radius,
        bitInfo=info,
        useFeatures=use_features,
        includeRedundantEnvironments=True
    )
    atom_envs = [None] * mol.GetNumAtoms()
    for atom_idx in range(mol.GetNumAtoms()):
        for bit, atom_list in info.items():
            for atom_info_tuple in atom_list:
                if atom_info_tuple[0] == atom_idx and atom_info_tuple[1] == radius:
                    atom_envs[atom_idx] = bit
                    break
            if atom_envs[atom_idx] is not None:
                break
    return atom_envs


## 2. Example: Cross-Environment Extraction
Let's extract input and target atom environments for a molecule.

In [5]:
smiles = sample_molecules['Caffeine']
input_envs = get_atom_envs(smiles, radius=0, use_features=False)
target_envs = get_atom_envs(smiles, radius=1, use_features=True)
print('Caffeine:')
print(' Input envs (radius=0, no features):', input_envs)
print(' Target envs (radius=1, features): ', target_envs)


Caffeine:
 Input envs (radius=0, no features): [2246728737, 2092489639, 3218693969, 2041434490, 3217380708, 3217380708, 3217380708, 864942730, 2092489639, 3217380708, 864942730, 2092489639, 2246728737, 2246728737]
 Target envs (radius=1, features):  [3205496007, 594910150, 3764343670, 3764335747, 1541843420, 1541844807, 594640068, 3205496716, 594910150, 594640192, 3205496716, 594910150, 3205496007, 3205496007]


[18:01:52] DEPRECATION WARNING: please use MorganGenerator
[18:01:52] DEPRECATION WARNING: please use MorganGenerator


In [6]:
vocab_target.get(3217380708)

## 3. Use loaded vocabularies for encoding
We now convert both input and target atom environments to token IDs using the loaded vocabularies.

In [7]:
def encode_with_vocab(env, vocab):
    if env is None:
        return UNK_ID
    return vocab.get(int(env), UNK_ID)

input_tokens = [encode_with_vocab(env, vocab_input) for env in input_envs]
target_tokens = [encode_with_vocab(env, vocab_target) for env in target_envs]
print('Input tokens:', input_tokens)
print('Target tokens:', target_tokens)


Input tokens: [5, 9, 1, 8, 2, 2, 2, 6, 9, 2, 6, 9, 5, 5]
Target tokens: [73, 36, 58, 6, 65, 24, 59, 39, 36, 139, 39, 36, 73, 73]


## 4. Masked Language Modeling Setup
Randomly mask input tokens and set up prediction targets.

In [8]:
def create_crossenv_mlm_sample(input_tokens, target_tokens, mask_prob=0.15):
    input_tokens = np.array(input_tokens, dtype=np.int64)
    target_tokens = np.array(target_tokens, dtype=np.int64)
    masked_input = input_tokens.copy()
    labels = np.full_like(target_tokens, -100)
    mask = np.random.rand(len(input_tokens)) < mask_prob
    for i, m in enumerate(mask):
        if m:
            masked_input[i] = MASK_ID
            labels[i] = target_tokens[i]
    return masked_input.tolist(), labels.tolist(), mask.tolist()

masked_input, labels, mask = create_crossenv_mlm_sample(input_tokens, target_tokens)
print('Masked input:', masked_input)
print('Labels:', labels)
print('Mask:', mask)


Masked input: [170, 170, 170, 8, 2, 2, 2, 6, 170, 2, 6, 9, 5, 5]
Labels: [73, 36, 58, -100, -100, -100, -100, -100, 36, -100, -100, -100, -100, -100]
Mask: [True, True, True, False, False, False, False, False, True, False, False, False, False, False]


## 5. PyTorch Tensor Preparation
Convert everything to tensors for model input.

In [9]:
input_tensor = torch.tensor([masked_input], dtype=torch.long)
labels_tensor = torch.tensor([labels], dtype=torch.long)
attention_mask = torch.tensor([[1]*len(masked_input)], dtype=torch.long)
print('Input tensor:', input_tensor)
print('Labels tensor:', labels_tensor)
print('Attention mask:', attention_mask)


Input tensor: tensor([[170, 170, 170,   8,   2,   2,   2,   6, 170,   2,   6,   9,   5,   5]])
Labels tensor: tensor([[  73,   36,   58, -100, -100, -100, -100, -100,   36, -100, -100, -100,
         -100, -100]])
Attention mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


---
You can now extend this notebook to process batches, use real vocabularies, or save datasets for MolE pretraining.